# Diarization: clean full-video runner
This replaces the earlier notebook and all extra repair cells. Run from the top. It contains the tested pipeline code and the setup fixes; your existing private dataset is sufficient.

**Kaggle:** Import this notebook, attach your private video dataset, enable Internet and GPU T4 ×2, and enable the existing `HF_TOKEN` secret. Keep the notebook private. A queued cell is waiting on Kaggle resources; this notebook cannot shorten that queue.

**Laptop:** Open this notebook in Jupyter/VS Code with Python 3.12. Set `DATA_DIR` below to the project folder or extracted bundle. CPU is supported. Dependencies install into a separate environment.

The runner first checks the two-minute video, then runs the full video if `RUN_FULL_VIDEO` is True. Download results and checkpoints before ending a cloud session. Attribution rules are unchanged from the tagged milestone. The complete cloud run is still unverified.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, time, zipfile

# Local: set this to the folder containing video.mp4 and the target .npy files.
# Kaggle: leave None; the attached dataset is found automatically.
DATA_DIR = None
RUN_FULL_VIDEO = True
BATCH_SIZE = 4
ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    # Fail before lengthy installation if this session has no GPU.
    probe = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
    if probe is None or probe.returncode or 'GPU ' not in probe.stdout:
        raise RuntimeError('No GPU in this session. Enable GPU T4 x2 in Settings, then rerun this cell.')
    print(probe.stdout)
    matches = list(Path('/kaggle/input').rglob('video.mp4'))
    if DATA_DIR is None:
        if len(matches) != 1:
            raise RuntimeError('Attach the private video dataset; expected one video.mp4, or set DATA_DIR explicitly.')
        DATA_DIR = matches[0].parent
    BASE = Path('/kaggle/working')
else:
    # Convenience for the current local project; otherwise use the chosen folder.
    local_project = Path('/home/think/projects/whisperx_diarization')
    DATA_DIR = DATA_DIR or (local_project if local_project.exists() else Path.cwd())
    BASE = Path.cwd()/'diarization-run'
DATA = Path(DATA_DIR).expanduser().resolve()
for name in ('video.mp4', 'longer_2min.mp4', 'voice_embeddings.npy', 'face_embeddings.npy'):
    if not (DATA/name).is_file():
        raise RuntimeError(f'Missing {name}. Set DATA_DIR to the project or extracted dataset folder.')
BASE.mkdir(parents=True, exist_ok=True)
WORK = BASE/'diarization'
WORK.mkdir(exist_ok=True)
RESULTS = BASE/'results'
RESULTS.mkdir(exist_ok=True)
CACHE = BASE/'stage-cache'
VENV = BASE/'diarization-venv'
PYTHON = str(VENV/('Scripts/python.exe' if os.name == 'nt' else 'bin/python'))
print('Input folder:', DATA)
print('Results folder:', RESULTS)
print('Setup will use:', PYTHON)

## Install and verify the environment
No activation or separate `wrapt`/ONNX repair cells are needed. Setup checks whether pip works, rather than only checking that Python exists. It uses virtualenv's bundled pip when creation or repair is needed. Installing dependencies can take several minutes; progress appears below.

In [ ]:
EMBEDDED_FILES = {'chainofrules.py': '"""Evidence-based refactor of the recovered runnable chainofrules.py.\nRun from the existing project directory with HF_TOKEN set in the environment.\nAccepts any video and matching target embedding files.\nAll rows in each embedding file are reference samples of the same target.\nDefaults retain short.mp4, large-v2, ECAPA and buffalo_l.\nConfidence values are heuristic evidence strengths, not calibrated probabilities.\n"""\nimport os\nimport argparse\nimport json\nimport re\nimport gc\nfrom pathlib import Path\nfrom importlib.metadata import version\nimport pandas as pd\nimport onnxruntime as ort\nfrom cloud_runtime import StageCache, file_digest, create_face_analyzer\nfrom dataclasses import dataclass, field, asdict\nimport cv2\nimport numpy as np\nimport torch\nimport torchaudio\nimport whisperx\nfrom whisperx.diarize import DiarizationPipeline\nfrom speechbrain.inference.speaker import SpeakerRecognition\nfrom insightface.app import FaceAnalysis\nfrom collections import defaultdict\n\n\n@dataclass(frozen=True)\nclass Baseline:\n    raw_speaker_track: str\n    speaker: str\n\n@dataclass(frozen=True)\nclass Evidence:\n    source: str\n    target_score: float\n    confidence: float\n    details: dict = field(default_factory=dict)\n\n@dataclass\nclass TimelineSegment:\n    start: float\n    end: float\n    text: str\n    baseline: Baseline\n    words: list = field(default_factory=list)\n    evidence: list = field(default_factory=list)\n    final_speaker: str = "Uncertain"\n    final_confidence: float = 0.0\n    reasons: list = field(default_factory=list)\n\n\ndef normalize_vector(value):\n    value = np.asarray(value, dtype=np.float32).reshape(-1)\n    norm = np.linalg.norm(value)\n    if not np.all(np.isfinite(value)) or norm <= 0:\n        raise ValueError("Embedding must be finite and nonzero")\n    return value / norm\n\n\ndef short_voice_crop(segment, previous=None, following=None, media_duration=None):\n    """Recover small timing gaps without including neighboring utterances."""\n    start, end = segment.start, segment.end\n    if end - start < 0.4:\n        lower = previous.end if previous is not None else 0.0\n        upper = following.start if following is not None else media_duration\n        start = max(lower, start - 0.15)\n        end = min(end + 0.15, upper) if upper is not None else end\n        # Overlapping transcript boundaries are not safe padding opportunities.\n        if start > segment.start or end < segment.end:\n            return segment.start, segment.end\n    return start, end\n\n\ndef add_question_response_evidence(segment, previous, tracks, target_track, mapping_confidence):\n    """A conversational hypothesis, never a police-specific identity rule."""\n    if previous is None or segment.end - segment.start > 1.0:\n        return\n    gap = segment.start - previous.end\n    if not 0.0 <= gap <= 0.6 or mapping_confidence < 0.75:\n        return\n    if previous.final_speaker in ("Uncertain", "NonTarget_Unknown", "Unknown_Speaker"):\n        return\n    if previous.final_confidence < 0.65:\n        return\n    question = previous.text.strip().lower()\n    # Narrow to addressed yes/no questions; punctuation alone is insufficient.\n    direct_question = re.match(\n        r"^(?:(?:ok|okay|all right)[.,]?\\s+)?"\n        r"(?:do you|did you|have you|are you|were you|can you|could you|"\n        r"would you|will you|don\'t you|didn\'t you|haven\'t you|aren\'t you)\\b", question)\n    if not question.endswith("?") or direct_question is None:\n        return\n    answer = re.sub(r"[^a-z\' ]", " ", segment.text.lower()).split()\n    if not answer or len(answer) > 4 or answer[0] not in ("yes", "no", "yeah", "yep", "nope", "nah"):\n        return\n    question_track = target_track if previous.final_speaker == "Target_Speaker" else previous.final_speaker\n    if question_track not in tracks:\n        return\n    candidates = sorted(track for track in tracks if track != question_track)\n    candidate = candidates[0] if len(candidates) == 1 else None\n    segment.evidence.append(Evidence("question_response", 0.0, 0.20,\n        {"question_start": previous.start, "question_track": question_track,\n         "candidate_tracks": candidates, "candidate_track": candidate,\n         "gap": gap, "assumption": "Immediate brief answer may be a different speaker; not voice-verified."}))\n\n\n\n\ndef add_brief_exchange_evidence(segment, previous, tracks, target_track, mapping_confidence):\n    """Tentative acknowledgement or confirmation, with independent voice agreement."""\n    if previous is None or mapping_confidence < .75 or segment.end - segment.start >= .4:\n        return\n    if not 0 <= segment.start - previous.end <= .6:\n        return\n    words = re.findall(r"[a-z\']+", segment.text.lower())\n    preceding = re.findall(r"[a-z\']+", previous.text.lower())\n    acknowledgement = words in (["okay"], ["ok"], ["oh", "okay"], ["oh", "ok"])\n    confirmation = (preceding in (["really"], ["seriously"]) and previous.text.strip().endswith("?")\n                    and words in (["yes"], ["yeah"], ["yep"], ["no"], ["nope"]))\n    if not acknowledgement and not confirmation:\n        return\n    previous_track = target_track if previous.final_speaker == "Target_Speaker" else previous.final_speaker\n    candidates = sorted(set(tracks) - {previous_track})\n    if previous_track not in tracks or len(candidates) != 1:\n        return\n    if acknowledgement and (previous.final_confidence < .65 or previous.text.strip().endswith("?")):\n        return\n    if confirmation:\n        # A weak question is usable only when its baseline agrees, a target face is\n        # tracked, and it was not itself attributed through conversational inference.\n        face = next((e for e in previous.evidence if e.source == "target_face_visible"), None)\n        if (previous.final_confidence < .25 or previous.baseline.raw_speaker_track != previous_track\n            or previous_track != target_track or face is None\n            or not face.details.get("target_visible_hint", False)\n            or any("inference" in reason for reason in previous.reasons)):\n            return\n    voice = next((e for e in segment.evidence if e.source == "local_voice"), None)\n    profiles = voice.details.get("track_similarities", {}) if voice is not None else {}\n    if (voice is None or voice.confidence > .30 or len(profiles) < 2\n        or max(profiles.values()) >= .30 or voice.details.get("best_track") != candidates[0]):\n        return\n    segment.evidence.append(Evidence("question_response", 0, .20,\n        {"candidate_track": candidates[0], "previous_track": previous_track,\n         "assumption": "Brief acknowledgement or confirmation may change speaker; weak voice agrees, not verified."}))\n\n\ndef add_echo_question_evidence(segment, previous, tracks, target_track, mapping_confidence):\n    """A brief quoted question can suggest another speaker, never establish one."""\n    if previous is None or not segment.text.strip().endswith("?"):\n        return\n    tokens = lambda text: re.findall(r"[a-z\']+", text.lower())\n    phrase, statement = tokens(segment.text), tokens(previous.text)\n    if not 2 <= len(phrase) <= 5 or statement[-len(phrase):] != phrase:\n        return\n    if not 0 <= segment.start - previous.end <= 0.8 or segment.end - segment.start > 1.2:\n        return\n    if previous.final_confidence < 0.65 or mapping_confidence < 0.75:\n        return\n    previous_track = target_track if previous.final_speaker == "Target_Speaker" else previous.final_speaker\n    candidates = sorted(set(tracks) - {previous_track})\n    if previous_track not in tracks or len(candidates) != 1:\n        return\n    segment.evidence.append(Evidence("echo_question", 0, 0.20,\n        {"candidate_track": candidates[0], "previous_track": previous_track,\n         "assumption": "Brief repeated question may come from the listener; not voice-verified."}))\n\n\ndef bbox_iou(left, right):\n    x1, y1 = max(left[0], right[0]), max(left[1], right[1])\n    x2, y2 = min(left[2], right[2]), min(left[3], right[3])\n    intersection = max(0.0, x2-x1) * max(0.0, y2-y1)\n    left_area = max(0.0, left[2]-left[0]) * max(0.0, left[3]-left[1])\n    right_area = max(0.0, right[2]-right[0]) * max(0.0, right[3]-right[1])\n    return intersection / max(left_area + right_area - intersection, 1e-9)\n\n\ndef collect_visual_evidence(segment, cap, fps, face_analyzer, target_face_centroid):\n    """Track a recently recognized face through head turns; mouth motion is only a hint."""\n    if not np.isfinite(fps) or fps <= 0:\n        segment.evidence.append(Evidence("visual_context", 0, 0, {"reason": "invalid_fps"}))\n        return\n    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    # Lead-in frames establish identity; only frames inside speech measure mouth motion.\n    times = np.arange(max(0.0, segment.start - 0.5), segment.end, 0.125)\n    indices = np.unique(np.rint(times * fps).astype(int))\n    best, anchor_best, direct_matches, frames_read = None, None, 0, 0\n    anchor = None\n    observations, apertures, target_frames = [], [], []\n    for index in indices:\n        if frame_count > 0 and index >= frame_count:\n            continue\n        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))\n        ok, frame = cap.read()\n        if not ok:\n            continue\n        frames_read += 1\n        time = index / fps\n        candidates = []\n        for face in face_analyzer.get(frame):\n            embedding = getattr(face, "embedding", None)\n            if embedding is None:\n                continue\n            embedding = normalize_vector(embedding)\n            similarity = float(np.dot(target_face_centroid, embedding))\n            if segment.start <= time <= segment.end:\n                best = similarity if best is None else max(best, similarity)\n            candidates.append((similarity, face, embedding))\n        recognized = [item for item in candidates if item[0] >= 0.40]\n        selected, identity_source = None, None\n        if recognized:\n            selected = max(recognized, key=lambda item: item[0])\n            direct_matches += 1\n            anchor_best = selected[0] if anchor_best is None else max(anchor_best, selected[0])\n            identity_source = "reference_match"\n        elif anchor is not None and time - anchor[2] <= 0.30:\n            linked = [item for item in candidates\n                      if bbox_iou(item[1].bbox, anchor[0]) >= 0.20\n                      and float(np.dot(item[2], anchor[1])) >= 0.45]\n            if linked:\n                selected = max(linked, key=lambda item: float(np.dot(item[2], anchor[1])))\n                identity_source = "face_continuity"\n        for similarity, face, embedding in candidates:\n            observations.append({"frame": int(index), "similarity": similarity,\n                                 "bbox": face.bbox.tolist()})\n        if selected is None:\n            continue\n        similarity, face, embedding = selected\n        anchor = (face.bbox.copy(), embedding, time)\n        if not segment.start <= time <= segment.end:\n            continue\n        target_frames.append({"frame": int(index), "identity_source": identity_source,\n                              "similarity": similarity})\n        landmarks = getattr(face, "landmark_3d_68", None)\n        if landmarks is not None and np.all(np.isfinite(landmarks)):\n            # Standard 68-point inner mouth: aperture / width in 3D landmark coordinates.\n            width = float(np.linalg.norm(landmarks[60] - landmarks[64]))\n            if width > 1e-6:\n                apertures.append(float(np.linalg.norm(landmarks[62] - landmarks[66]) / width))\n    spread = float(np.percentile(apertures, 90) - np.percentile(apertures, 10)) if len(apertures) >= 5 else 0.0\n    motion_hint = direct_matches >= 2 and len(apertures) >= 5 and spread >= 0.03\n    segment.evidence.append(Evidence("target_face_visible", 0, 0,\n        {"best_similarity": best, "identity_anchor_similarity": anchor_best,\n         "target_visible_hint": bool(target_frames), "tracked_target_frames": target_frames,\n         "active_speaker_verified": False}))\n    segment.evidence.append(Evidence("visual_context", 0, 0,\n        {"frames_read": frames_read, "observations": observations,\n         "note": "Face identity and visibility do not identify police or prove speech."}))\n    segment.evidence.append(Evidence("target_mouth_motion", 1.0 if motion_hint else 0.0,\n        0.20 if motion_hint else 0.0,\n        {"direct_identity_matches": direct_matches, "mouth_samples": len(apertures),\n         "aperture_spread": spread, "active_speaker_verified": False,\n         "note": "Weak landmark motion hint; no lipreading or audio-visual synchronization model."}))\n\n\ndef resolve_segment(segment, target_track, mapping_confidence):\n    """Only the resolver assigns final identity; visibility alone cannot flip it."""\n    raw = segment.baseline.raw_speaker_track\n    known = raw != "Unknown_Speaker"\n    prior_weight = 0.55 * mapping_confidence if known else 0.0\n    prior = 1.0 if raw == target_track else -1.0\n    score, weight = prior * prior_weight, prior_weight\n    reasons = [f"baseline={raw}; mapping strength={mapping_confidence:.3f}"]\n    voice = None\n    response = None\n    mouth_motion = None\n    echo = None\n    visible = None\n    for item in segment.evidence:\n        # Presence/context describes the scene, not the active speaker.\n        if item.source in ("target_face_visible", "visual_context"):\n            if item.source == "target_face_visible":\n                visible = item\n            continue\n        if item.source == "echo_question":\n            echo = item\n            continue\n        if item.source == "target_mouth_motion":\n            mouth_motion = item\n            continue\n        if item.source == "question_response":\n            response = item\n            continue\n        contribution = item.target_score * item.confidence\n        score += contribution\n        weight += item.confidence\n        if item.source == "local_voice":\n            voice = item\n        if item.confidence:\n            reasons.append(f"{item.source}: {contribution:+.3f}")\n    normalized = score / max(weight, 1e-9)\n    # Role phrases cannot establish or contradict identity without acoustic support.\n    acoustic_support = prior_weight > 0.08 or (voice is not None and voice.confidence > 0.2)\n    strong_conflict = (voice is not None and voice.confidence >= 0.5\n                      and voice.target_score * prior < -0.4 and known)\n    # A strong reference match plus an independent track match can correct diarization.\n    details = voice.details if voice is not None else {}\n    matched_track = details.get("best_track")\n    verified_correction = (strong_conflict and voice.confidence >= 0.5\n                          and abs(voice.target_score) >= 0.4\n                          and details.get("track_margin", 0.0) >= 0.10\n                          and matched_track is not None\n                          and details.get("track_similarities", {}).get(matched_track, -1.0) >= 0.30\n                          and ((voice.target_score > 0 and matched_track == target_track)\n                               or (voice.target_score < 0 and matched_track != target_track)))\n    insufficient_short_audio = (segment.end - segment.start < 0.4\n                                and (voice is None or voice.confidence == 0))\n    response_track = response.details.get("candidate_track") if response is not None else None\n    profiles = details.get("track_similarities", {})\n    weak_padded_voice = (segment.end - segment.start < 0.4 and voice is not None\n                         and voice.confidence <= 0.30 and len(profiles) >= 2\n                         and max(profiles.values()) < 0.30)\n    response_inference = ((insufficient_short_audio or weak_padded_voice) and response_track is not None\n                          and response.confidence > 0)\n    visual_inference = (mouth_motion is not None and mouth_motion.confidence > 0\n                        and voice is not None and voice.confidence > 0\n                        and mapping_confidence >= 0.75\n                        and len(details.get("track_similarities", {})) >= 2\n                        and details.get("track_margin", 1.0) < 0.05\n                        and max(details["track_similarities"].values()) < 0.35)\n    echo_inference = (echo is not None and echo.details["candidate_track"] == target_track\n                      and visible is not None and visible.details.get("target_visible_hint", False)\n                      and len(profiles) >= 2 and max(profiles.values()) < 0.30\n                      and matched_track == target_track and voice.confidence < 0.5)\n    if echo_inference:\n        final = "Target_Speaker"\n        reasons.append("weak repeated-question inference with face continuity and weak supporting voice profile; not voice-verified")\n    elif visual_inference:\n        final = "Target_Speaker"\n        reasons.append("weak visible-target mouth-motion inference; independent voice profiles are ambiguous")\n    elif response_inference:\n        final = "Target_Speaker" if response_track == target_track else response_track\n        reasons.append(f"weak question/answer inference to {response_track}; not voice-verified")\n    elif verified_correction:\n        final = "Target_Speaker" if matched_track == target_track else matched_track\n        reasons.append(f"independent voice profile supports correction to {matched_track}")\n    elif insufficient_short_audio or not acoustic_support or abs(normalized) <= 0.18 or strong_conflict:\n        final = "Uncertain"\n        reasons.append("weak, balanced, or conflicting acoustic evidence")\n    elif normalized > 0:\n        final = "Target_Speaker"\n    else:\n        final = raw if known and raw != target_track else "NonTarget_Unknown"\n    segment.final_speaker = final\n    # Avoid baseline-only certainty and account for weak global separation.\n    segment.final_confidence = float(min(abs(normalized), weight / 1.55))\n    if verified_correction:\n        segment.final_confidence = float(min(voice.confidence, abs(voice.target_score),\n                                             details["track_margin"] / 0.20))\n    if echo_inference:\n        segment.final_confidence = 0.20\n    elif visual_inference:\n        segment.final_confidence = 0.25\n    elif response_inference:\n        segment.final_confidence = min(0.25, response.confidence)\n    elif insufficient_short_audio:\n        segment.final_confidence = 0.0\n        reasons.append("short utterance has no usable local voice evidence")\n    if segment.end - segment.start < 0.4:\n        segment.final_confidence = min(segment.final_confidence, 0.30)\n    segment.reasons = reasons\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Resolve a supplied target voice in any video.")\n    parser.add_argument("video", nargs="?", default="short.mp4")\n    parser.add_argument("--voice-priors", default="voice_embeddings.npy")\n    parser.add_argument("--face-priors", default="face_embeddings.npy")\n    parser.add_argument("--output", default="diarization_evidence.json")\n    parser.add_argument("--batch-size", type=int, default=16)\n    parser.add_argument("--cache-dir", help="Reuse completed stages for matching inputs/code/runtime")\n    args = parser.parse_args()\n    if args.batch_size < 1:\n        parser.error("--batch-size must be positive")\n    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")\n    if not HF_TOKEN:\n        raise RuntimeError("Set HF_TOKEN (or HUGGINGFACE_TOKEN) before running.")\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    compute_type = "float16" if torch.cuda.is_available() else "int8"\n\n    # =====================================================================\n    # 1. CORE PIPELINE INITIALIZATION\n    # =====================================================================\n    print("⏳ Initializing Core Tracking Engines...")\n    def release_gpu():\n        gc.collect()\n        if device == "cuda":\n            torch.cuda.empty_cache()\n\n    fingerprint = {"inputs": {name: file_digest(path) for name, path in\n        (("video", args.video), ("voice", args.voice_priors), ("face", args.face_priors))},\n        "code": file_digest(__file__), "runtime_helper": file_digest(Path(__file__).with_name("cloud_runtime.py")),\n        "device": device, "batch_size": args.batch_size,\n        "versions": {name: version(name) for name in\n            ("torch", "torchaudio", "whisperx", "speechbrain", "insightface", "numpy")},\n        "onnxruntime": ort.__version__, "providers": ort.get_available_providers()}\n    cache = StageCache(args.cache_dir, fingerprint)\n    print(f"WhisperX/SpeechBrain device: {device}")\n\n    # Load Priors Matrix\n    voice_priors = np.load(args.voice_priors, allow_pickle=False)\n    face_priors = np.load(args.face_priors, allow_pickle=False)\n    if voice_priors.ndim == 1: voice_priors = np.expand_dims(voice_priors, axis=0)\n    if face_priors.ndim == 1: face_priors = np.expand_dims(face_priors, axis=0)\n\n    def reference_centroid(samples, label):\n        if samples.ndim != 2:\n            raise ValueError(f"{label}: expected a vector or matrix of target samples")\n        norms = np.linalg.norm(samples, axis=1)\n        valid = np.all(np.isfinite(samples), axis=1) & (norms > 0)\n        if not np.any(valid):\n            raise ValueError(f"{label}: no valid target samples")\n        normalized = samples[valid] / norms[valid, None]\n        print(f"{label}: using {len(normalized)} of {len(samples)} target reference samples")\n        return normalize_vector(np.mean(normalized, axis=0))\n\n    target_voice_vector = reference_centroid(voice_priors, "Voice references")\n    target_face_centroid = reference_centroid(face_priors, "Face references")\n\n    # =====================================================================\n    # 2. AUDIO & VIDEO DATA PREP\n    # =====================================================================\n    print("⏳ Preparing media data tracks...")\n    video_path = args.video\n    audio_loaded = whisperx.load_audio(video_path)\n    cap = cv2.VideoCapture(video_path)\n    fps = cap.get(cv2.CAP_PROP_FPS)\n\n    waveform, sample_rate = torchaudio.load(video_path)\n    if sample_rate != 16000:\n        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)\n        waveform = resampler(waveform)\n    waveform = torch.mean(waveform, dim=0, keepdim=True)\n    total_samples = waveform.shape[1]\n\n    # =====================================================================\n    # 3. GENERAL TRANSCRIPTION & LAYERING\n    # =====================================================================\n    print("⏳ Processing WhisperX Text Script...")\n    def transcribe():\n        model = whisperx.load_model("large-v2", device, compute_type=compute_type)\n        try:\n            return model.transcribe(audio_loaded, batch_size=args.batch_size)\n        finally:\n            del model\n            release_gpu()\n\n    asr_result = cache.get("transcription", transcribe)\n\n    def align():\n        model, metadata = whisperx.load_align_model(language_code=asr_result["language"], device=device)\n        try:\n            return whisperx.align(asr_result["segments"], model, metadata, audio_loaded,\n                                  device, return_char_alignments=False)\n        finally:\n            del model\n            release_gpu()\n\n    aligned_result = cache.get("alignment", align)\n    print("⏳ Generating Unsupervised Voice Tracks...")\n\n    def diarize():\n        model = DiarizationPipeline(token=HF_TOKEN, device=device)\n        try:\n            # Preserve original whole-video track IDs; no independent chunk clustering.\n            return model(audio_loaded)[["start", "end", "speaker"]].to_dict(orient="records")\n        finally:\n            del model\n            release_gpu()\n\n    diarize_segments = pd.DataFrame(cache.get("diarization", diarize))\n    embedding_model = SpeakerRecognition.from_hparams(\n        source="speechbrain/spkrec-ecapa-voxceleb", savedir="pretrained_models/spkrec-ecapa-voxceleb",\n        run_opts={"device": device})\n    face_analyzer, face_providers = create_face_analyzer(device, ort, FaceAnalysis)\n\n    target_voice_vector = normalize_vector(target_voice_vector)\n    target_face_centroid = normalize_vector(target_face_centroid)\n    embedding_cache = {}\n\n    def audio_embedding(start, end):\n        key = (float(start), float(end))\n        if key in embedding_cache:\n            return embedding_cache[key]\n        left = max(0, int(start * 16000))\n        right = min(total_samples, int(end * 16000))\n        checkpoint_name = f"voice_{left}_{right}"\n        saved = cache.read(checkpoint_name)\n        if saved is not None:\n            result = np.asarray(saved, dtype=np.float32)\n            embedding_cache[key] = result\n            return result\n        result = None\n        if right - left >= 6400:\n            try:\n                with torch.no_grad():\n                    result = normalize_vector(embedding_model.encode_batch(\n                        waveform[:, left:right].to(device)).flatten().cpu().numpy())\n                if result.shape != target_voice_vector.shape:\n                    raise ValueError("Voice prior dimensions do not match ECAPA output")\n            except ValueError:\n                raise\n            except Exception as exc:\n                print(f"Voice embedding unavailable at {start:.2f}-{end:.2f}: {exc}")\n        if result is not None:\n            cache.write(checkpoint_name, result.tolist())\n        embedding_cache[key] = result\n        return result\n\n    cluster_scores = defaultdict(list)\n    cluster_embeddings = defaultdict(list)\n    for _, row in diarize_segments.iterrows():\n        start, end = float(row["start"]), float(row["end"])\n        if end - start < 0.6:\n            continue\n        emb = audio_embedding(start, end)\n        if emb is not None:\n            track = str(row["speaker"])\n            cluster_scores[track].append(float(np.dot(target_voice_vector, emb)))\n            cluster_embeddings[track].append((start, end, emb))\n    means = {track: float(np.mean(scores)) for track, scores in cluster_scores.items()}\n    ranked = sorted(means, key=means.get, reverse=True)\n    target_track = ranked[0] if ranked else None\n    target_mean = means[target_track] if ranked else 0.0\n    # No invented competitor when only one cluster has usable speech.\n    other_mean = means[ranked[1]] if len(ranked) > 1 else None\n    separation = target_mean - other_mean if other_mean is not None else 0.0\n    mapping_confidence = min(1.0, max(0.0, separation / 0.15))\n    print("\\n--- Baseline voice affinity ---")\n    for track in ranked:\n        print(f"{track}: {means[track]:.3f} ({len(cluster_scores[track])} samples)")\n    print(f"Target candidate: {target_track}; mapping strength={mapping_confidence:.3f}")\n\n    assigned = whisperx.assign_word_speakers(diarize_segments, aligned_result)\n    timeline = []\n    for source in assigned["segments"]:\n        raw = str(source.get("speaker") or "Unknown_Speaker")\n        base = "Target_Speaker" if raw == target_track else ("Unknown" if raw == "Unknown_Speaker" else raw)\n        timeline.append(TimelineSegment(float(source["start"]), float(source["end"]),\n                        source["text"].strip(), Baseline(raw, base), source.get("words", [])))\n\n    def collect_voice(segment, previous=None, following=None):\n        crop_start, crop_end = short_voice_crop(segment, previous, following, total_samples / 16000)\n        emb = audio_embedding(crop_start, crop_end)\n        similarity = float(np.dot(target_voice_vector, emb)) if emb is not None else None\n        strength = min(1.0, max(0.0, (segment.end - segment.start) / 1.2)) * mapping_confidence\n        if segment.end - segment.start < 0.4:\n            strength = min(strength, 0.30)\n        target_score = 0.0\n        if similarity is not None and separation >= 0.03:\n            target_score = float(np.clip((similarity - (target_mean + other_mean) / 2) / separation, -1, 1))\n        else:\n            strength = 0.0\n        # Exclude intersecting speech from profiles so a segment cannot validate itself.\n        track_similarities = {}\n        profile_counts = {}\n        if emb is not None:\n            for track, samples in cluster_embeddings.items():\n                independent = [vector for start, end, vector in samples\n                               if end <= crop_start or start >= crop_end]\n                if len(independent) < 2:\n                    continue\n                centroid = normalize_vector(np.mean(independent, axis=0))\n                track_similarities[track] = float(np.dot(emb, centroid))\n                profile_counts[track] = len(independent)\n        candidates = sorted(track_similarities, key=track_similarities.get, reverse=True)\n        best_track = candidates[0] if len(candidates) >= 2 else None\n        margin = (track_similarities[candidates[0]] - track_similarities[candidates[1]]\n                  if len(candidates) >= 2 else 0.0)\n        segment.evidence.append(Evidence("local_voice", target_score, strength,\n            {"similarity": similarity, "crop_start": crop_start, "crop_end": crop_end,\n             "target_mean": target_mean, "competitor_mean": other_mean,\n             "track_similarities": track_similarities, "independent_profile_counts": profile_counts,\n             "best_track": best_track, "track_margin": margin}))\n\n    def collect_visual(segment):\n        name = f"visual_{segment.start:.6f}_{segment.end:.6f}"\n        saved = cache.read(name)\n        if saved is not None:\n            segment.evidence.extend(Evidence(**item) for item in saved)\n            return\n        offset = len(segment.evidence)\n        collect_visual_evidence(segment, cap, fps, face_analyzer, target_face_centroid)\n        cache.write(name, [asdict(item) for item in segment.evidence[offset:]])\n\n    def collect_semantic(segment):\n        # Without an explicit role-to-identity mapping, words cannot identify a person.\n        # Keep semantic/context observations neutral for arbitrary videos and targets.\n        segment.evidence.append(Evidence("semantic_context", 0.0, 0.0,\n            {"identity_mapping": None,\n             "note": "No role or phrase is assumed to identify the supplied target."}))\n\n    try:\n        all_tracks = {str(track) for track in diarize_segments["speaker"].dropna().unique()}\n        for index, segment in enumerate(timeline):\n            previous = timeline[index - 1] if index else None\n            following = timeline[index + 1] if index + 1 < len(timeline) else None\n            collect_voice(segment, previous, following)\n            collect_visual(segment)\n            collect_semantic(segment)\n            add_question_response_evidence(segment, previous, all_tracks, target_track, mapping_confidence)\n            add_brief_exchange_evidence(segment, previous, all_tracks, target_track, mapping_confidence)\n            add_echo_question_evidence(segment, previous, all_tracks, target_track, mapping_confidence)\n            resolve_segment(segment, target_track, mapping_confidence)\n            print(f"Resolved segment {index + 1}/{len(timeline)} at {segment.end:.1f}s", flush=True)\n        print("\\n--- Evidence-Based Speaker Resolution ---")\n        for segment in timeline:\n            print(f"[{segment.start:.2f}s - {segment.end:.2f}s] {segment.final_speaker} "\n                  f"(strength={segment.final_confidence:.2f}): {segment.text}")\n            print(f"    baseline={segment.baseline.speaker}; raw={segment.baseline.raw_speaker_track}")\n            for item in segment.evidence:\n                print(f"    {item.source}: score={item.target_score:+.2f}, strength={item.confidence:.2f}, {item.details}")\n        with open(args.output, "w", encoding="utf-8") as output:\n            json.dump({"target_candidate": target_track, "cluster_voice_means": means,\n                       "mapping_strength": mapping_confidence,\n                       "runtime": {"device": device, "face_providers": face_providers},\n                       "confidence_is_calibrated": False,\n                       "segments": [asdict(segment) for segment in timeline]}, output, indent=2, ensure_ascii=False)\n    finally:\n        cap.release()\n\n\nif __name__ == "__main__":\n    main()\n', 'cloud_runtime.py': '"""Small runtime helpers; attribution rules do not live here."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\n\ndef file_digest(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\nclass StageCache:\n    """Atomic, JSON-only checkpoints, isolated by inputs/code/runtime fingerprint."""\n    def __init__(self, directory, fingerprint):\n        self.root = None\n        if directory:\n            key = hashlib.sha256(json.dumps(fingerprint, sort_keys=True).encode()).hexdigest()\n            self.root = Path(directory) / key\n            self.root.mkdir(parents=True, exist_ok=True)\n            self.write("manifest", fingerprint)\n\n    def read(self, name):\n        if self.root is None:\n            return None\n        path = self.root / (name + ".json")\n        if not path.exists():\n            return None\n        return json.loads(path.read_text())\n\n    def write(self, name, value):\n        if self.root is None:\n            return\n        path = self.root / (name + ".json")\n        temporary = path.with_suffix(".tmp")\n        temporary.write_text(json.dumps(value, ensure_ascii=False))\n        os.replace(temporary, path)\n\n    def get(self, name, compute):\n        value = self.read(name)\n        if value is not None:\n            print(f"Reusing checkpoint: {name}")\n            return value\n        value = compute()\n        self.write(name, value)\n        return value\n\n\ndef create_face_analyzer(device, ort, factory):\n    if device == "cuda" and hasattr(ort, "preload_dlls"):\n        ort.preload_dlls()\n    use_cuda = device == "cuda" and "CUDAExecutionProvider" in ort.get_available_providers()\n    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if use_cuda else ["CPUExecutionProvider"]\n    analyzer = factory(name="buffalo_l", providers=providers)\n    analyzer.prepare(ctx_id=0 if use_cuda else -1, det_size=(640, 640))\n    actual = {name: model.session.get_providers() for name, model in analyzer.models.items()\n              if getattr(model, "session", None) is not None}\n    print(f"Face analysis actual providers: {actual}")\n    if device == "cuda" and (not actual or any("CUDAExecutionProvider" not in value for value in actual.values())):\n        print("WARNING: one or more face models are using CPU; check onnxruntime-gpu/CUDA libraries.")\n    return analyzer, actual\n', 'test_cloud_runtime.py': 'import tempfile\nimport unittest\nfrom types import SimpleNamespace\nfrom cloud_runtime import StageCache, create_face_analyzer\n\n\nclass CloudRuntimeTests(unittest.TestCase):\n    def test_cache_reuses_completed_stage_and_isolates_changed_inputs(self):\n        with tempfile.TemporaryDirectory() as directory:\n            cache = StageCache(directory, {"video": "one", "code": "one"})\n            self.assertEqual(cache.get("stage", lambda: {"speaker": "SPEAKER_02"}), {"speaker": "SPEAKER_02"})\n            repeated = StageCache(directory, {"code": "one", "video": "one"})\n            self.assertEqual(repeated.get("stage", lambda: self.fail("stage reran")), {"speaker": "SPEAKER_02"})\n            self.assertIsNone(StageCache(directory, {"video": "two", "code": "one"}).read("stage"))\n            self.assertIsNone(StageCache(directory, {"video": "one", "code": "two"}).read("stage"))\n            self.assertFalse(list(cache.root.glob("*.tmp")))\n\n    def test_disabled_cache_does_not_save(self):\n        cache = StageCache(None, {})\n        cache.write("stage", {"ok": True})\n        self.assertIsNone(cache.read("stage"))\n\n    def test_face_provider_selection_and_actual_fallback(self):\n        for device, available, expected, context in (\n            ("cpu", ["CUDAExecutionProvider", "CPUExecutionProvider"], ["CPUExecutionProvider"], -1),\n            ("cuda", ["CPUExecutionProvider"], ["CPUExecutionProvider"], -1),\n            ("cuda", ["CUDAExecutionProvider", "CPUExecutionProvider"], ["CUDAExecutionProvider", "CPUExecutionProvider"], 0)):\n            calls = []\n            def factory(**kwargs):\n                calls.append(kwargs)\n                return SimpleNamespace(prepare=lambda **options: calls.append(options),\n                    models={"recognition": SimpleNamespace(session=SimpleNamespace(get_providers=lambda: ["CPUExecutionProvider"]))})\n            ort = SimpleNamespace(get_available_providers=lambda: available, preload_dlls=lambda: None)\n            _, actual = create_face_analyzer(device, ort, factory)\n            self.assertEqual(calls[0]["providers"], expected)\n            self.assertEqual(calls[1]["ctx_id"], context)\n            self.assertEqual(actual["recognition"], ["CPUExecutionProvider"])\n\n    def test_pipeline_reuses_stages_voice_and_visual_evidence(self):\n        import contextlib\n        import io\n        import json\n        from pathlib import Path\n        from unittest.mock import patch, Mock\n        import numpy as np\n        import torch\n        import chainofrules as pipeline\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            (root/"video.mp4").write_bytes(b"fake media")\n            np.save(root/"voice.npy", np.ones((2, 192), dtype=np.float32))\n            np.save(root/"face.npy", np.ones((2, 512), dtype=np.float32))\n            records = [{"start": 0., "end": 1., "speaker": "SPEAKER_00"},\n                       {"start": 1., "end": 2., "speaker": "SPEAKER_01"}]\n            assigned = {"segments": [{"start": 0., "end": 1., "text": "Hello.", "speaker": "SPEAKER_00"}]}\n            whisper = Mock(); whisper.transcribe.return_value = {"language": "en", "segments": []}\n            voice = Mock(); voice.encode_batch.return_value = torch.ones((1, 1, 192))\n            detector = Mock(return_value=pipeline.pd.DataFrame(records))\n            cap = Mock(); cap.get.return_value = 30\n            argv = ["chainofrules.py", str(root/"video.mp4"), "--voice-priors", str(root/"voice.npy"),\n                    "--face-priors", str(root/"face.npy"), "--output", str(root/"result.json"),\n                    "--cache-dir", str(root/"cache")]\n            with patch("sys.argv", argv), patch.dict("os.environ", {"HF_TOKEN": "test-placeholder"}), \\\n                 patch.object(pipeline.torch.cuda, "is_available", return_value=False), \\\n                 patch.object(pipeline.whisperx, "load_audio", return_value=np.zeros(32000)), \\\n                 patch.object(pipeline.torchaudio, "load", return_value=(torch.zeros(1, 32000), 16000)), \\\n                 patch.object(pipeline.cv2, "VideoCapture", return_value=cap), \\\n                 patch.object(pipeline.whisperx, "load_model", return_value=whisper) as load, \\\n                 patch.object(pipeline.whisperx, "load_align_model", return_value=(Mock(), {})) as align_load, \\\n                 patch.object(pipeline.whisperx, "align", return_value=assigned), \\\n                 patch.object(pipeline.whisperx, "assign_word_speakers", return_value=assigned), \\\n                 patch.object(pipeline, "DiarizationPipeline", return_value=detector) as diarize_load, \\\n                 patch.object(pipeline.SpeakerRecognition, "from_hparams", return_value=voice) as voice_load, \\\n                 patch.object(pipeline, "create_face_analyzer", return_value=(Mock(), {})), \\\n                 patch.object(pipeline, "collect_visual_evidence") as visual, \\\n                 contextlib.redirect_stdout(io.StringIO()):\n                pipeline.main()\n                first = json.loads((root/"result.json").read_text())\n                pipeline.main()\n                second = json.loads((root/"result.json").read_text())\n                self.assertEqual(first, second)\n                self.assertEqual(load.call_count, 1)\n                self.assertEqual(align_load.call_count, 1)\n                self.assertEqual(diarize_load.call_count, 1)\n                self.assertEqual(voice.encode_batch.call_count, 2)\n                self.assertEqual(visual.call_count, 1)\n                self.assertEqual(voice_load.call_args.kwargs["run_opts"], {"device": "cpu"})\n                self.assertEqual(cap.release.call_count, 2)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'test_short_answers.py': '"""Behavior checks for conversational attribution, independent of model downloads."""\nimport unittest\nimport numpy as np\nfrom chainofrules import (Baseline, Evidence, TimelineSegment, short_voice_crop,\n                          add_question_response_evidence, add_echo_question_evidence, add_brief_exchange_evidence, resolve_segment, collect_visual_evidence)\n\n\nclass ShortAnswerTests(unittest.TestCase):\n    def question(self, text="Do you have any weapons?", strength=0.9, speaker="SPEAKER_00"):\n        segment = TimelineSegment(10.0, 12.83, text, Baseline(speaker, speaker))\n        segment.final_speaker, segment.final_confidence = speaker, strength\n        return segment\n\n    def reply(self, start=12.89, end=12.99, text="No.", raw="SPEAKER_00"):\n        segment = TimelineSegment(start, end, text, Baseline(raw, raw))\n        segment.evidence.append(Evidence("local_voice", 0.0, 0.0))\n        return segment\n\n    def infer(self, reply, question, tracks=("SPEAKER_00", "SPEAKER_01"), mapping=1.0):\n        add_question_response_evidence(reply, question, set(tracks), "SPEAKER_01", mapping)\n        resolve_segment(reply, "SPEAKER_01", mapping)\n        return reply\n\n    def test_brief_answer_has_weak_alternative_identity(self):\n        reply = self.infer(self.reply(), self.question())\n        self.assertEqual(reply.final_speaker, "Target_Speaker")\n        self.assertLessEqual(reply.final_confidence, 0.25)\n        self.assertTrue(any("not voice-verified" in reason for reason in reply.reasons))\n\n    def test_target_question_does_not_force_target_answer(self):\n        reply = self.infer(self.reply(raw="SPEAKER_01"), self.question(speaker="Target_Speaker"))\n        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n\n    def test_three_speakers_leave_answer_unresolved(self):\n        reply = self.infer(self.reply(), self.question(), ("SPEAKER_00", "SPEAKER_01", "SPEAKER_02"))\n        self.assertEqual(reply.final_speaker, "Uncertain")\n\n    def test_no_question_or_weak_question_leaves_answer_unresolved(self):\n        for question in (None, self.question("You are on private property."),\n                         self.question("Why are you here?"), self.question(strength=0.35)):\n            with self.subTest(question=question):\n                self.assertEqual(self.infer(self.reply(), question).final_speaker, "Uncertain")\n\n    def test_gap_overlap_and_long_answer_are_not_inferred(self):\n        for reply in (self.reply(13.5, 13.6), self.reply(12.8, 12.9),\n                      self.reply(12.89, 14.5), self.reply(text="No one should be doing that.")):\n            with self.subTest(reply=reply):\n                result = self.infer(reply, self.question())\n                self.assertFalse(any(item.source == "question_response" for item in result.evidence))\n                if reply.end - reply.start < 0.4:\n                    self.assertEqual(result.final_speaker, "Uncertain")\n\n    def test_weak_target_mapping_does_not_infer(self):\n        self.assertEqual(self.infer(self.reply(), self.question(), mapping=0.4).final_speaker, "Uncertain")\n\n    def test_available_voice_is_not_overridden_by_conversation(self):\n        reply = self.reply()\n        reply.evidence = [Evidence("local_voice", -1.0, 0.5)]\n        self.assertEqual(self.infer(reply, self.question()).final_speaker, "SPEAKER_00")\n\n    def test_padded_short_audio_does_not_claim_high_confidence(self):\n        reply = self.reply(raw="SPEAKER_01")\n        reply.evidence = [Evidence("local_voice", -0.64, 0.28)]\n        resolve_segment(reply, "SPEAKER_01", 1.0)\n        self.assertLessEqual(reply.final_confidence, 0.30)\n\n    def test_independent_voice_corrects_short_baseline_conflict(self):\n        reply = self.reply(start=0.25, end=0.92, raw="SPEAKER_01", text="Question")\n        reply.evidence = [Evidence("local_voice", -0.57, 0.55,\n            {"best_track": "SPEAKER_00", "track_margin": 0.15,\n             "track_similarities": {"SPEAKER_00": 0.31, "SPEAKER_01": 0.16}})]\n        resolve_segment(reply, "SPEAKER_01", 1.0)\n        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n\n    def test_poor_profile_match_cannot_correct_identity(self):\n        reply = self.reply(start=0.25, end=0.92, raw="SPEAKER_01", text="Question")\n        reply.evidence = [Evidence("local_voice", -0.8, 0.55,\n            {"best_track": "SPEAKER_00", "track_margin": 0.15,\n             "track_similarities": {"SPEAKER_00": 0.20, "SPEAKER_01": 0.05}})]\n        resolve_segment(reply, "SPEAKER_01", 1.0)\n        self.assertEqual(reply.final_speaker, "Uncertain")\n\n    def test_face_presence_alone_does_not_override_voice(self):\n        reply = self.reply(start=28.0, end=29.0)\n        reply.evidence = [Evidence("local_voice", -0.8, 0.7,\n            {"track_margin": 0.03, "track_similarities": {"SPEAKER_00": 0.23, "SPEAKER_01": 0.20}}),\n            Evidence("target_face_visible", 1.0, 1.0)]\n        resolve_segment(reply, "SPEAKER_01", 1.0)\n        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n\n    def test_mouth_hint_is_tentative_only_with_ambiguous_profiles(self):\n        for margin, expected in ((0.03, "Target_Speaker"), (0.20, "SPEAKER_00")):\n            reply = self.reply(start=28.0, end=29.0)\n            reply.evidence = [Evidence("local_voice", -0.8, 0.7,\n                {"track_margin": margin, "track_similarities": {"SPEAKER_00": 0.23, "SPEAKER_01": 0.20}}),\n                Evidence("target_mouth_motion", 1.0, 0.2)]\n            resolve_segment(reply, "SPEAKER_01", 1.0)\n            self.assertEqual(reply.final_speaker, expected)\n            if margin < 0.05:\n                self.assertLessEqual(reply.final_confidence, 0.25)\n\n    def test_face_identity_continues_through_head_turn_but_not_bbox_jump(self):\n        class Capture:\n            index = 0\n            def get(self, prop): return 20\n            def set(self, prop, value): self.index = int(value)\n            def read(self): return True, np.zeros((200, 200, 3), dtype=np.uint8)\n        class Face:\n            pass\n        for jump in (False, True):\n            capture = Capture()\n            class Analyzer:\n                def get(self, frame):\n                    face = Face()\n                    face.embedding = np.array([0.7, np.sqrt(1-0.7**2)]) if capture.index < 2 else np.array([0.3, np.sqrt(1-0.3**2)])\n                    face.bbox = np.array([10,10,80,100]) if not jump or capture.index < 2 else np.array([120,120,190,200])\n                    points = np.zeros((68,3))\n                    points[64,0] = 10\n                    points[66,1] = 0.4 if capture.index % 2 else 1.2\n                    face.landmark_3d_68 = points\n                    return [face]\n            reply = self.reply(start=0.5, end=1.4)\n            collect_visual_evidence(reply, capture, 8.0, Analyzer(), np.array([1.0, 0.0]))\n            motion = next(item for item in reply.evidence if item.source == "target_mouth_motion")\n            self.assertEqual(motion.confidence > 0, not jump)\n\n    def test_crop_respects_both_neighbors(self):\n        following = TimelineSegment(13.01, 15.6, "Next", Baseline("SPEAKER_00", "SPEAKER_00"))\n        start, end = short_voice_crop(self.reply(), self.question(), following, 30.0)\n        self.assertAlmostEqual(start, 12.83)\n        self.assertAlmostEqual(end, 13.01)\n        self.assertLess(end-start, 0.4)\n\n    def test_overlapping_timing_does_not_trim_reply(self):\n        preceding = self.question()\n        preceding.end = 12.92\n        reply = self.reply()\n        self.assertEqual(short_voice_crop(reply, preceding, None, 30.0), (reply.start, reply.end))\n\n\n    def test_echo_requires_weak_supporting_voice_and_visible_target(self):\n        previous = self.question("You are being arrested for criminal loitering.")\n        reply = self.reply(start=13.3, end=13.88, text="Criminal loitering?")\n        reply.evidence = [Evidence("local_voice", -.8, .48,\n            {"best_track": "SPEAKER_01", "track_margin": .07,\n             "track_similarities": {"SPEAKER_00": .05, "SPEAKER_01": .12}}),\n            Evidence("target_face_visible", 0, 0, {"target_visible_hint": True})]\n        add_echo_question_evidence(reply, previous, {"SPEAKER_00", "SPEAKER_01"}, "SPEAKER_01", 1)\n        resolve_segment(reply, "SPEAKER_01", 1)\n        self.assertEqual(reply.final_speaker, "Target_Speaker")\n        self.assertEqual(reply.final_confidence, .20)\n        reply.evidence = [e for e in reply.evidence if e.source != "target_face_visible"]\n        resolve_segment(reply, "SPEAKER_01", 1)\n        self.assertNotEqual(reply.final_speaker, "Target_Speaker")\n\n    def test_echo_does_not_override_clear_voice_or_choose_among_three_people(self):\n        previous = self.question("You are being arrested for criminal loitering.")\n        reply = self.reply(start=13.3, end=13.88, text="Criminal loitering?")\n        reply.evidence = [Evidence("local_voice", -1, .6,\n            {"best_track": "SPEAKER_00", "track_margin": .4,\n             "track_similarities": {"SPEAKER_00": .5, "SPEAKER_01": .1}}),\n            Evidence("target_face_visible", 0, 0, {"target_visible_hint": True})]\n        add_echo_question_evidence(reply, previous, {"SPEAKER_00", "SPEAKER_01"}, "SPEAKER_01", 1)\n        resolve_segment(reply, "SPEAKER_01", 1)\n        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n        reply.evidence = []\n        add_echo_question_evidence(reply, previous, {"SPEAKER_00", "SPEAKER_01", "SPEAKER_02"}, "SPEAKER_01", 1)\n        self.assertEqual(reply.evidence, [])\n\n    def test_padded_but_weak_voice_allows_tentative_answer(self):\n        reply = self.reply()\n        reply.evidence = [Evidence("local_voice", -1, .28,\n            {"best_track": "SPEAKER_00", "track_similarities": {"SPEAKER_00": .23, "SPEAKER_01": .08}})]\n        self.infer(reply, self.question())\n        self.assertEqual(reply.final_speaker, "Target_Speaker")\n        self.assertEqual(reply.final_confidence, .20)\n\n\n    def test_acknowledgement_requires_independent_agreement(self):\n        previous = self.question("Your request has been accepted.")\n        reply = self.reply(text="Oh, okay.")\n        reply.evidence = [Evidence("local_voice", -.7, .28,\n            {"best_track": "SPEAKER_01", "track_similarities": {"SPEAKER_00": 0, "SPEAKER_01": .1}})]\n        add_brief_exchange_evidence(reply, previous, {"SPEAKER_00", "SPEAKER_01"}, "SPEAKER_01", 1)\n        resolve_segment(reply, "SPEAKER_01", 1)\n        self.assertEqual(reply.final_speaker, "Target_Speaker")\n        self.assertEqual(reply.final_confidence, .2)\n        for strength, best, tracks in ((.6, "SPEAKER_01", {"SPEAKER_00", "SPEAKER_01"}),\n                                      (.28, "SPEAKER_00", {"SPEAKER_00", "SPEAKER_01"}),\n                                      (.28, "SPEAKER_01", {"SPEAKER_00", "SPEAKER_01", "SPEAKER_02"})):\n            reply.evidence = [Evidence("local_voice", -.7, strength,\n                {"best_track": best, "track_similarities": {"SPEAKER_00": 0, "SPEAKER_01": .1}})]\n            add_brief_exchange_evidence(reply, previous, tracks, "SPEAKER_01", 1)\n            self.assertEqual(len(reply.evidence), 1)\n\n    def test_confirmation_requires_face_baseline_and_no_inference_chain(self):\n        previous = self.question("Really?", .30, "SPEAKER_01")\n        previous.final_speaker = "Target_Speaker"\n        previous.evidence = [Evidence("target_face_visible", 0, 0, {"target_visible_hint": True})]\n        reply = self.reply(text="Yeah.", raw="SPEAKER_01")\n        voice = Evidence("local_voice", -1, .15,\n            {"best_track": "SPEAKER_00", "track_similarities": {"SPEAKER_00": .06, "SPEAKER_01": .02}})\n        reply.evidence = [voice]\n        add_brief_exchange_evidence(reply, previous, {"SPEAKER_00", "SPEAKER_01"}, "SPEAKER_01", 1)\n        resolve_segment(reply, "SPEAKER_01", 1)\n        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n        self.assertEqual(reply.final_confidence, .20)\n        for face, reasons in (([], []), (previous.evidence, ["weak question/answer inference"])):\n            previous.evidence, previous.reasons = face, reasons\n            reply.evidence = [voice]\n            add_brief_exchange_evidence(reply, previous, {"SPEAKER_00", "SPEAKER_01"}, "SPEAKER_01", 1)\n            self.assertEqual(len(reply.evidence), 1)\n\n\nif __name__ == "__main__":\n    unittest.main()\n'}
for name, content in EMBEDDED_FILES.items():
    (WORK/name).write_text(content)

# Strip notebook-only display settings from every child process.
ENV = os.environ.copy()
ENV['MPLBACKEND'] = 'Agg'
ENV['MPLCONFIGDIR'] = str(BASE/'matplotlib')
ENV['PYTHONDONTWRITEBYTECODE'] = '1'
ENV['PYTHONUNBUFFERED'] = '1'
ENV.pop('PYTHONPATH', None)

def checked(args, **kwargs):
    return subprocess.run(args, env=ENV, check=True, **kwargs)

print('1/4: Creating/checking environment', flush=True)
ready = False
if Path(PYTHON).exists():
    ready = subprocess.run([PYTHON, '-m', 'pip', '--version'], env=ENV,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not ready:
    # Install bootstrap tools outside the runtime, without replacing notebook packages.
    bootstrap = BASE/'bootstrap-tools'
    checked([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap),
             'virtualenv>=20.26,<21', 'wrapt'])
    bootstrap_env = ENV.copy()
    bootstrap_env['PYTHONPATH'] = str(bootstrap)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-download', str(VENV)],
                   env=bootstrap_env, check=True)
checked([PYTHON, '-m', 'pip', '--version'])
print('2/4: Installing pipeline dependencies (including wrapt in the venv)', flush=True)
requirements = [
    'whisperx==3.8.6', 'speechbrain==1.1.1', 'insightface==2.0',
    'torch==2.8.0', 'torchaudio==2.8.0', 'numpy==2.5.3',
    'opencv-python==5.0.0.93', 'onnxruntime-gpu==1.23.2', 'wrapt']
checked([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
checked([PYTHON, '-m', 'pip', 'install', *requirements])
# InsightFace's metadata requires the CPU-named distribution, even though GPU
# supplies the same import. Check dependency resolution before removing that overlap.
print('3/4: Removing overlapping ONNX packages and installing CUDA 12 build', flush=True)
checked([PYTHON, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'])
checked([PYTHON, '-m', 'pip', 'install', '--no-deps', '--force-reinstall',
         'onnxruntime-gpu==1.23.2'])
if shutil.which('ffmpeg') is None:
    raise RuntimeError('Install ffmpeg on this laptop and rerun setup. Kaggle normally includes it.')
library_dirs = subprocess.check_output([PYTHON, '-c',
    "import site,pathlib; print(':'.join(str(p) for d in site.getsitepackages() for p in pathlib.Path(d).glob('nvidia/*/lib')))"], env=ENV, text=True).strip()
ENV['LD_LIBRARY_PATH'] = library_dirs + ':' + ENV.get('LD_LIBRARY_PATH', '')
print('4/4: Verifying imports, GPU and behavior checks', flush=True)
verification = """import torch,onnxruntime as ort,wrapt
import chainofrules
print('Torch:', torch.__version__, 'CUDA build:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (local CPU)')
print('ONNX:', ort.__version__, ort.__file__)
print('Advertised providers:', ort.get_available_providers())
"""
if ON_KAGGLE:
    verification += "assert torch.cuda.is_available(), 'Kaggle GPU is not available to this runtime'\nassert 'CUDAExecutionProvider' in ort.get_available_providers(), 'GPU ONNX package missing'"
checked([PYTHON, '-c', verification], cwd=WORK)
checked([PYTHON, '-m', 'unittest', 'test_cloud_runtime', 'test_short_answers'], cwd=WORK)
print('Setup complete. Continue to credentials and tests.', flush=True)

## Credentials and optional checkpoint restore
The token is never embedded in this notebook or printed. Kaggle reads the authorized secret; on a laptop, use the environment variable or the hidden prompt. If restoring checkpoints, attach the downloaded archive as another private dataset. Completed stages with an exact fingerprint are reused; interrupted stages rerun.

In [ ]:
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    ENV['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
else:
    import getpass
    ENV['HF_TOKEN'] = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN') or getpass.getpass('Hugging Face token: ')
if not ENV['HF_TOKEN']:
    raise RuntimeError('A token with diarization-model access is required.')
if ON_KAGGLE:
    for archive in Path('/kaggle/input').rglob('stage-checkpoints.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (BASE/member.filename).resolve()
                if not target.is_relative_to(CACHE.resolve()):
                    raise RuntimeError('Unexpected checkpoint archive path')
            zipped.extractall(BASE)
print('Credentials configured; token not displayed.')

In [ ]:
def export_checkpoints():
    if CACHE.exists():
        shutil.make_archive(str(BASE/'stage-checkpoints'), 'zip', CACHE.parent, CACHE.name)

def run_test(video_name, stem, batch_size=4):
    output = RESULTS/(stem+'_evidence.json')
    command = [PYTHON, str(WORK/'chainofrules.py'), str(DATA/video_name),
               '--voice-priors', str(DATA/'voice_embeddings.npy'),
               '--face-priors', str(DATA/'face_embeddings.npy'),
               '--output', str(output), '--cache-dir', str(CACHE),
               '--batch-size', str(batch_size)]
    started = time.monotonic()
    try:
        with (RESULTS/(stem+'.log')).open('w') as log:
            process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                # Never save or display the secret even if a dependency prints it.
                line = line.replace(ENV['HF_TOKEN'], '[REDACTED]')
                log.write(line); log.flush()
                print(line if len(line) < 1000 else line[:1000]+' ... [full line saved in log]\n', end='')
            if process.wait() != 0:
                raise RuntimeError('Test failed; see the log. For CUDA out-of-memory, retry with batch_size=1.')
    finally:
        export_checkpoints()
    result = json.loads(output.read_text())
    transcript = '\n'.join(f"[{s['start']:.2f}-{s['end']:.2f}] {s['final_speaker']} (strength={s['final_confidence']:.2f}): {s['text']}" for s in result['segments'])
    (RESULTS/(stem+'_transcript.txt')).write_text(transcript+'\n')
    with (RESULTS/'runtime-packages.txt').open('w') as packages:
        checked([PYTHON, '-m', 'pip', 'freeze'], stdout=packages)
    print(f'Elapsed: {(time.monotonic()-started)/60:.1f} minutes')
    print('Runtime:', result.get('runtime'))
    return result

## Two-minute validation, then full video
On Kaggle this stops if any actual face-model session falls back to CPU. An advertised CUDA provider alone does not prove GPU inference works. Review labels with weak strengths in both results; longer-video clustering can differ from the short clip.

In [ ]:
two_minute = run_test('longer_2min.mp4', 'two_minute', BATCH_SIZE)
providers = two_minute.get('runtime', {}).get('face_providers', {})
if ON_KAGGLE and (not providers or any('CUDAExecutionProvider' not in p for p in providers.values())):
    raise RuntimeError('Actual face analysis fell back to CPU. Full video has not started. Paste the provider warning from the log.')
print((RESULTS/'two_minute_transcript.txt').read_text())
if RUN_FULL_VIDEO:
    print('Two-minute validation completed; starting whole-video diarization.', flush=True)
    full_video = run_test('video.mp4', 'full_video', BATCH_SIZE)
else:
    print('Full run disabled. Set RUN_FULL_VIDEO=True and rerun this cell to continue; matching stages are cached.')

## Save results
Download both archives on Kaggle. Session files and installed environments are temporary; the notebook itself does not retain model caches or outputs unless you save/download them. On your laptop these paths remain on disk.

In [ ]:
from IPython.display import FileLink, display
export_checkpoints()
shutil.make_archive(str(BASE/'diarization-results'), 'zip', RESULTS)
display(FileLink(str(BASE/'diarization-results.zip')))
display(FileLink(str(BASE/'stage-checkpoints.zip')))
print('Saved in:', BASE)